# 04 · Eleição do vencedor e promoção

**Feature F8** (critério 8) · dono: ver `config/equipe.yaml` · funções em `src/promocao.py`

| | |
|---|---|
| **Entradas** | `resultados_competicao.json` e os modelos de `models/candidatos/<cycle_id>/` (notebook 03), `teste.parquet` (notebook 01), `models/modelo_vigente.json` e o arquivo do vigente em `models/registro/` (no Actions, recuperados do armazenamento persistente), `cfg['promocao']` |
| **Saídas** | `models/registro/modelo_<cycle_id>_<A\|B>.joblib` (vencedor do ciclo, comprimido) · `models/modelo_vigente.json` atualizado · `evidencias/promocao/decisao_<cycle_id>.json` · run `04_promocao` no MLflow |

**Regras**
1. Entre A e B vence a maior média de `f1_macro` no CV do dev (números da F5). Empate: menor desvio entre os folds.
2. O vencedor e o vigente são avaliados no **teste deste ciclo**, os mesmos registros para os dois.
3. Sem vigente, o vencedor é promovido se passar no piso. Com vigente, só é promovido se passar no piso **e** superar o vigente em pelo menos `min_melhoria_abs`. Senão, o vigente é mantido.
4. O vencedor de todo ciclo é guardado, promovido ou não. Nenhum modelo anterior é apagado.
5. Qualquer inconsistência levanta erro **antes** de gravar o manifesto: o notebook falha e nada é promovido.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))

from src.config import carregar_config, carregar_contexto, resolver
from src.preprocessamento import carregar_particao
from src import promocao

cfg = carregar_config()
ctx = carregar_contexto(cfg, RAIZ)  # escrito pelo notebook 01
print(f"cycle_id={ctx['cycle_id']} | data_version={ctx['data_version']}")

cycle_id=20261120T012057Z-local | data_version=dv018-f95c3940


## 1. Eleição entre A e B

A regra foi fixada antes de rodar, em `config.yaml` (`promocao`).

In [2]:
regra = cfg["promocao"]
print("Regra:", {k: regra[k] for k in ("metrica", "direcao", "min_melhoria_abs", "piso_metrica")})

caminho_resultados = resolver(cfg["dados"]["dir_processed"], RAIZ) / "resultados_competicao.json"
resultados = json.loads(caminho_resultados.read_text(encoding="utf-8"))
promocao.validar_resultados(resultados, ctx, cfg)

metrica = regra["metrica"]
display(pd.DataFrame({
    nome: {
        "algoritmo": item["algoritmo"],
        f"{metrica} (média CV)": round(item["cv"]["media"][metrica], 4),
        f"{metrica} (desvio CV)": round(item["cv"]["desvio"][metrica], 4),
    }
    for nome, item in resultados["modelos"].items()
}).T.rename_axis("candidato"))

vencedor = promocao.eleger_vencedor(resultados, cfg)
item_vencedor = resultados["modelos"][vencedor]
print(f"Vencedor do ciclo: modelo {vencedor} ({item_vencedor['algoritmo']})")

Regra: {'metrica': 'f1_macro', 'direcao': 'maximizar', 'min_melhoria_abs': 0.005, 'piso_metrica': 0.6}


,algoritmo,f1_macro (média CV),f1_macro (desvio CV)
candidato,,,
A,logistic_regression,0.7406,0.0133
B,random_forest,0.7783,0.0447


Vencedor do ciclo: modelo B (random_forest)


## 2. Vencedor contra o vigente, no teste do ciclo

O teste é a parte mais recente dos dados do ciclo e não participou do treino nem do CV. O vigente é o arquivo salvo no ciclo em que foi promovido, sem retreino.

In [3]:
import joblib

manifesto = promocao.carregar_manifesto(cfg, RAIZ)
if any(h["cycle_id"] == ctx["cycle_id"] for h in manifesto["historico"]):
    raise ValueError(f"O ciclo {ctx['cycle_id']} já foi decidido. Rode o notebook 01 para abrir um ciclo novo.")

teste = carregar_particao("teste", cfg, RAIZ)
tempo = cfg["avaliacao"]["coluna_tempo"]
print(f"teste: {len(teste):,} linhas, de {teste[tempo].min():%Y-%m-%d} a {teste[tempo].max():%Y-%m-%d}")

modelo_vencedor = joblib.load(RAIZ / item_vencedor["caminho_modelo"])
metricas_vencedor = promocao.avaliar_no_teste(modelo_vencedor, teste, cfg)

modelo_vigente, vigente = promocao.carregar_vigente(manifesto, RAIZ)
metricas_vigente = None if vigente is None else promocao.avaliar_no_teste(modelo_vigente, teste, cfg)

colunas = {f"vencedor {vencedor} (ciclo atual)": metricas_vencedor}
if vigente is None:
    print("Ainda não há modelo vigente: primeiro ciclo.")
else:
    colunas[f"vigente {vigente['modelo']} (ciclo {vigente['cycle_id']})"] = metricas_vigente
display(pd.DataFrame(colunas).round(4).rename_axis("métrica no teste do ciclo"))

teste: 15,783 linhas, de 2016-09-24 a 2016-12-31


Ainda não há modelo vigente: primeiro ciclo.


,vencedor B (ciclo atual)
métrica no teste do ciclo,
f1_macro,0.6773
balanced_accuracy,0.6759
recall_cancelamento,0.3642
precision_cancelamento,0.9481
roc_auc,0.8040
pr_auc,0.7738


## 3. Decisão

In [4]:
decisao = promocao.decidir_promocao(
    {"metricas_teste": metricas_vencedor},
    None if vigente is None else {"metricas_teste": metricas_vigente},
    cfg,
)
print(decisao["decisao"].upper(), "-", decisao["motivo"])

PROMOVIDO - primeiro ciclo sem vigente; f1_macro=0.6773 passa no piso 0.6


## 4. Registro: modelo, MLflow e manifesto

O vencedor ganha uma cópia comprimida em `models/registro/`. O workflow (F7) envia essa pasta para a Release do GitHub. O manifesto só é gravado depois que o run do MLflow foi registrado.

In [5]:
from src import rastreio

arquivo = promocao.guardar_modelo(modelo_vencedor, ctx["cycle_id"], vencedor, RAIZ)
print("modelo guardado:", arquivo.relative_to(RAIZ), f"({arquivo.stat().st_size / 2**20:.1f} MB)")

runs_ciclo = rastreio.historico_ciclos(cfg, RAIZ, cycle_id=ctx["cycle_id"])
runs_treino = runs_ciclo[runs_ciclo["etapa"] == f"03_modelo_{vencedor}"]
run_id_treino = runs_treino["run_id"].iloc[-1] if not runs_treino.empty else None
if run_id_treino is None:
    print(f"AVISO: run 03_modelo_{vencedor} deste ciclo não encontrado no MLflow")

registro = {
    "cycle_id": ctx["cycle_id"],
    "data_version": ctx["data_version"],
    "decidido_em": promocao._agora_iso(),
    **decisao,
    "candidato": {
        "modelo": vencedor,
        "algoritmo": item_vencedor["algoritmo"],
        "params": item_vencedor["params"],
        "seed": item_vencedor["seed"],
        "features": resultados["features_eleitas"],
        "metricas_cv": item_vencedor["cv"]["media"],
        "metricas_teste": metricas_vencedor,
        "arquivo": arquivo.name,
        "run_id_treino": run_id_treino,
    },
    "vigente_anterior": None if vigente is None else {
        "cycle_id": vigente["cycle_id"],
        "modelo": vigente["modelo"],
        "arquivo": vigente["arquivo"],
        "metricas_teste": metricas_vigente,
    },
    "teste": {"n": len(teste), "inicio": str(teste[tempo].min().date()), "fim": str(teste[tempo].max().date())},
}

with rastreio.iniciar_run("04_promocao", ctx, cfg, RAIZ) as run:
    rastreio.registrar_parametros({
        "promocao": {k: regra[k] for k in ("metrica", "direcao", "min_melhoria_abs", "piso_metrica")},
        "vencedor": vencedor,
        "algoritmo_vencedor": item_vencedor["algoritmo"],
        "vigente_anterior": "nenhum" if vigente is None else vigente["cycle_id"],
    })
    rastreio.registrar_metricas({f"teste_vencedor_{m}": v for m, v in metricas_vencedor.items()})
    if metricas_vigente is not None:
        rastreio.registrar_metricas({f"teste_vigente_{m}": v for m, v in metricas_vigente.items()})
    rastreio.registrar_features(resultados["features_eleitas"])
    rastreio.registrar_decisao(registro)
registro["run_id_promocao"] = run.info.run_id

manifesto = promocao.atualizar_manifesto(registro, cfg, RAIZ)
evidencia = promocao.salvar_evidencia_decisao(registro, RAIZ)
print("run 04_promocao:", registro["run_id_promocao"])
print("manifesto:", promocao.caminho_manifesto(cfg, RAIZ).relative_to(RAIZ))
print("evidência:", evidencia.relative_to(RAIZ))
vig = manifesto["vigente"]
print("vigente agora:", "nenhum" if vig is None else f"modelo {vig['modelo']} do ciclo {vig['cycle_id']} ({vig['arquivo']})")

modelo guardado: models\registro\modelo_20261120T012057Z-local_B.joblib (19.0 MB)


run 04_promocao: 18b69869ea874379b362a3f484c63bf9
manifesto: models\modelo_vigente.json
evidência: evidencias\promocao\decisao_20261120T012057Z-local.json
vigente agora: modelo B do ciclo 20261120T012057Z-local (modelo_20261120T012057Z-local_B.joblib)


## 5. Histórico de decisões

In [6]:
display(promocao.tabela_historico(manifesto).round(4))

,cycle_id,data_version,vencedor,algoritmo,valor_vencedor,vigente_anterior,valor_vigente,decisao
0,20261120T012057Z-local,dv018-f95c3940,B,random_forest,0.6773,None,None,promovido
